# Querying the PChProp collection in PSDI lakehouse

This notebook give a few examles of querying the PChProp collection through Trino.

## 1. Install and import dependencies

We will need the Trino Python client to fetch data and `pandas` for post-processing results.

In [ ]:
!pip install trino pandas

Now let's import the required components:

In [ ]:
import pandas as pd
from trino.dbapi import connect
from trino.auth import OAuth2Authentication

## 1. Connect to Trino

Run the cell below to open a connection. At the first run, you will be prompted to authenticate via the PSDI Authentication System in your browser.

In [ ]:
TRINO_HOST = "trino-staging.psdi.ac.uk"

try:
    conn = connect(
        host=TRINO_HOST,
        port=443,
        http_scheme="https",
        auth=OAuth2Authentication(),
        catalog="psdi",
        request_timeout=300,
    )
    
    cursor = conn.cursor()

    # Validate the connection
    cursor.execute("SHOW SCHEMAS FROM psdi")
    cursor.fetchone()
    
    print("\nConnected to Trino successfully!")

except Exception as e:
    print("\nFailed to connect to Trino")
    raise

---

### 2 What is SQL? 

[SQL (Structured Query Language)](https://en.wikipedia.org/wiki/SQL) is commonly used to query and analyze structured data stored in databases and lakehouses. It provides a standard way to retrieve, filter, aggregate, and organize data using declarative queries.

In SQL, keywords such as `SELECT`, `FROM`, `WHERE`, and `GROUP BY` are conventionally written in uppercase to improve readability. Table names, column names, and other identifiers are commonly written in lowercase. SQL queries are typically written across multiple lines to make them easier to read and understand.

For example:

```SQL
SELECT formula_pretty, bandgap
FROM materials_project.materials
WHERE bandgap > 1.0
LIMIT 5
```
**Note that the semicolon is not required at the end of SQL statements in Trino.**

This query:

- selects the columns `formula_pretty` and `bandgap`
- reads data from the `materials_project.materials` table
- filters rows where the `bandgap` is larger than 1.0
- returns only the first 5 matching rows



### 3 General pattern for executing commands

To execute SQL queries, you first need to create a **cursor object**, which represents an active SQL session. The recommended approach is to use a context manager:

```python
with conn.cursor() as cursor:
    # Send SQL command to Trino for execution
    cursor.execute("<SQL coomand>")  
    
    # Retrieve all results returned by Trino
    results = cursor.fetchall()
```

### 4 Listing available tables within PChProp

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SHOW TABLES FROM pchprop")
    
    tables = cursor.fetchall()
    print(tables)

Those tables can be referenced using dot notation, e.g., `pchprop.compounds`


### 5 Explore columns

To inspect column names and types in the specific table, you can run the `DESCRIBE` command:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("DESCRIBE pchprop.compounds")
    
    columns = cursor.fetchall()
    print(columns)

As you can see, the output of `DESCRIBE` is not particularly pretty. The following cell addresses this issue by going over all available tables, printing their data schemas, and providing a sample of the data.

In [ ]:
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

with conn.cursor() as cursor:
    # Get all tables in the schema
    cursor.execute("SHOW TABLES FROM pchprop")
    tables = [row[0] for row in cursor.fetchall()]

    print(f"Found {len(tables)} tables in pchprop: {tables}\n")

    # Loop over each table and show its schema
    for table in tables:
        cursor.execute(f'DESCRIBE pchprop."{table}"')
        columns = cursor.fetchall()

        schema_df = pd.DataFrame(
            columns,
            columns=["column_name", "type", "extra", "comment"][:len(columns[0])]
        )
        schema_df = schema_df.drop(columns=["extra", "comment"], errors="ignore")

        print("=" * 60)
        print(f"Table: pchprop.{table}")
        print("=" * 60)
        with pd.option_context("display.max_colwidth", None):
            display(schema_df)

        # Sample rows
        cursor.execute(f'SELECT * FROM pchprop."{table}" LIMIT 1')
        rows = cursor.fetchall()
        col_names = [desc[0] for desc in cursor.description]
        sample_df = pd.DataFrame(rows, columns=col_names)
    
        print("\n" + "=" * 60)
        print("SAMPLE ROWS")
        print("=" * 60)
        with pd.option_context("display.max_colwidth", None):
            display(sample_df)

### 6 Counting rows in a table

First let's count rows in the `compounds` table:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM pchprop.compounds")
    rows = cursor.fetchone()  # fetchone() returns a single tuple like (1234,)
    
    print("Row count:", rows[0])

Let's go over all the available tables in PChProp and count the rows in each one:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("SHOW TABLES FROM pchprop")
    tables = [row[0] for row in cursor.fetchall()]

    for table in tables:
        cursor.execute(f'SELECT COUNT(*) FROM pchprop."{table}"')
        rows = cursor.fetchone()
        print(f"Table '{table}' contains {rows[0]} rows.")

## 7. Example Queries

### 7.1 Find a compound by id

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT *
    FROM pchprop.compounds
    WHERE compound_id = 'PChProp-CMP-4c80f7a2526592e551e80a88'
    """)

    # Fetch column names and data
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()

    # Print rows as dictionaries
    for row in data:
        print(dict(zip(columns, row)))

### 7.2 Working with nested structures

The `identifiers` column in the `compounds` table is a nested column of Trino’s ROW type.

Individual fields can be accessed in Trino using dot notation:

- `identifiers.inchi`
- `identifiers.smiles`

Now let's find a compound by InChI:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT *
    FROM pchprop.compounds
    WHERE identifiers.inchi = 'InChI=1S/C8H7NO2/c1-2-7-4-3-5-8(6-7)9(10)11/h2-6H,1H2'
    """)

    # Fetch column names and data
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()

    # Print rows as dictionaries
    for row in data:
        print(dict(zip(columns, row)))

### 7.3 Retrieving a subset of columns

To do this, replace the asterisk after the `SELECT` keyword with a list of the required columns.

Here is an example in which only `compound_id`, `inchi` and `smiles` are retrieved.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT compound_id, identifiers.inchi, identifiers.smiles
    FROM pchprop.compounds
    WHERE identifiers.inchi = 'InChI=1S/C8H7NO2/c1-2-7-4-3-5-8(6-7)9(10)11/h2-6H,1H2'
    """)

    # Fetch column names and data
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()

    # Print rows as dictionaries
    for row in data:
        print(dict(zip(columns, row)))

### 7.4 Find solubility measurements with values between -2.489 and -2.487

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT *
    FROM pchprop.measurements
    WHERE property.key = 'solubility'
      AND value.parsed >= -2.4890
      AND value.parsed <= -2.4887
    """)

    # Fetch column names and data
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()

    print(f"Found {len(data)} row(s)")

    # Print rows as dictionaries
    for row in data:
        print(dict(zip(columns, row)))

### 7.5 Using Pandas for handling results

Here is an example of how pandas can be used to handle the results of the previous query:

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT *
        FROM pchprop.measurements
        WHERE property.key = 'solubility'
          AND value.parsed >= -2.4890
          AND value.parsed <= -2.4887
    """)

    columns = [column[0] for column in cursor.description]
    data = cursor.fetchall()

df = pd.DataFrame(data, columns=columns)

print(f"Found {len(df)} row(s)")
display(df)

Note that Trino ROW-type values are represented as dictionaries inside the pandas DataFrame. Pandas does not automatically expand these fields into separate columns. Individual fields can be accessed using their dictionary keys. 

The `pd.json_normalize()` function can expand these dictionaries into separate columns, making the data easier to filter, sort, and analyse. For example:

In [ ]:
df_flat = pd.json_normalize(df.to_dict("records"))
print(df_flat)

### 7.6 Joining tables

We can join the `compounds` and `measurements` tables on the field `compound_id`. As the output will be massive, let's limit it to a specific compound, defined by its InChI.

In [ ]:
with conn.cursor() as cursor:
    cursor.execute("""
    SELECT *
    FROM pchprop.compounds AS compounds
    JOIN pchprop.measurements AS measurements
        ON compounds.compound_id = measurements.compound_id
    WHERE identifiers.inchi = 'InChI=1S/C8H7NO2/c1-2-7-4-3-5-8(6-7)9(10)11/h2-6H,1H2'
    """)

    # Fetch column names and data
    columns = [col[0] for col in cursor.description]
    data = cursor.fetchall()

    print(f"Found {len(data)} row(s)")

    # Print rows as dictionaries
    for row in data:
        print(dict(zip(columns, row)))

Note that there are different types of joins in SQL. The default one is called INNER JOIN (INNER can be omitted).